# Отчет по лабораторной работе №3
## Тема: Арифметические вычисления

**Студент:** ИУ5-35Б Штыка Денис Александрович

**Вариант:** 23

**Выражение:** ((((a-c)*b)/c)*a)

## Задание 1
**Условие:** Вывести ASCII-код символа, переданного программе параметром командной строки.

**Исходный код (asm):**

```asm
format ELF64
public _start

section '.data' writable
    newline db 10

section '.bss' writable
    result_buf rb 32

section '.text' executable
_start:
    cmp qword [rsp], 2
    jne .error

    mov rsi, [rsp + 16]
    cmp byte [rsi + 1], 0
    jne .error
    movzx rax, byte [rsi]
    cmp rax, 127
    ja .error

    call print_number
    xor rdi, rdi
    jmp exit

.error:
    mov rdi, 1

exit:
    mov rax, 60
    syscall

print_number:
    mov r8, result_buf + 32
    mov r9, r8
    mov rcx, 10

.convert:
    xor rdx, rdx
    div rcx
    dec r8
    add dl, '0'
    mov [r8], dl
    test rax, rax
    jnz .convert

    mov rax, 1
    mov rdi, 1
    mov rsi, r8
    mov rdx, r9
    sub rdx, r8
    syscall

    mov rax, 1
    mov rdi, 1
    mov rsi, newline
    mov rdx, 1
    syscall
    ret
```

**Сборка и запуск:**

```bash
cd Lab_3
fasm task1.asm task1.o
ld -o task1 task1.o
./task1 A
```

**Результат работы программы:**

```text
65
```

## Задание 2
**Условие:** Вычислить выражение варианта на языке ассемблера. Значения `a`, `b`, `c` передаются параметрами командной строки.

Для варианта 23 получено выражение `((((a-c)*b)/c)*a)`. При целочисленном делении результат округляется к нулю.

**Исходный код (asm):**

```asm
format ELF64
public _start

section '.data' writable
    newline db 10

section '.bss' writable
    a_value rq 1
    b_value rq 1
    c_value rq 1
    result_buf rb 32

section '.text' executable
_start:
    cmp qword [rsp], 4
    jne .error

    mov rsi, [rsp + 16]
    call parse_integer
    mov [a_value], rax

    mov rsi, [rsp + 24]
    call parse_integer
    mov [b_value], rax

    mov rsi, [rsp + 32]
    call parse_integer
    mov [c_value], rax
    test rax, rax
    jz .error

    mov rax, [a_value]
    sub rax, [c_value]
    imul rax, [b_value]
    cqo
    idiv qword [c_value]
    imul rax, [a_value]
    call print_number

    xor rdi, rdi
    jmp exit

.error:
    mov rdi, 1

exit:
    mov rax, 60
    syscall

parse_integer:
    xor rax, rax
    mov r8, 1
    cmp byte [rsi], '-'
    jne .digits
    mov r8, -1
    inc rsi

.digits:
    movzx rdx, byte [rsi]
    test dl, dl
    jz .parsed
    sub rdx, '0'
    imul rax, rax, 10
    add rax, rdx
    inc rsi
    jmp .digits

.parsed:
    imul rax, r8
    ret

print_number:
    xor r10, r10
    test rax, rax
    jns .absolute
    mov r10, 1
    neg rax

.absolute:
    mov r8, result_buf + 32
    mov r9, r8
    mov rcx, 10

.convert:
    xor rdx, rdx
    div rcx
    dec r8
    add dl, '0'
    mov [r8], dl
    test rax, rax
    jnz .convert

    test r10, r10
    jz .write
    dec r8
    mov byte [r8], '-'

.write:
    mov rax, 1
    mov rdi, 1
    mov rsi, r8
    mov rdx, r9
    sub rdx, r8
    syscall

    mov rax, 1
    mov rdi, 1
    mov rsi, newline
    mov rdx, 1
    syscall
    ret
```

**Сборка и запуск:**

```bash
cd Lab_3
fasm task2.asm task2.o
ld -o task2 task2.o
./task2 8 2 4
```

**Результат работы программы:**

```text
16
```

Дополнительная проверка: `./task2 -5 3 2` выводит `50`. При `c=0` программа завершает работу с кодом `1`.

## Задание 3
**Условие:** Составить на языке C программу, выполняющую те же вычисления, и сравнить результаты.

**Исходный код (c):**

```c
#include <stdio.h>
#include <stdlib.h>

int main(int argc, char *argv[]) {
    if (argc != 4) {
        return 1;
    }

    long long a = strtoll(argv[1], NULL, 10);
    long long b = strtoll(argv[2], NULL, 10);
    long long c = strtoll(argv[3], NULL, 10);

    if (c == 0) {
        return 1;
    }

    long long result = (((a - c) * b) / c) * a;
    printf("%lld\n", result);
    return 0;
}
```

**Сборка и запуск:**

```bash
cd Lab_3
gcc -Wall -Wextra -Werror -std=c11 task3.c -o task3
./task3 8 2 4
```

**Результат работы программы:**

```text
16
```

Дополнительная проверка: `./task3 -5 3 2` выводит `50`. При `c=0` программа завершает работу с кодом `1`.

## Вывод
В контейнере с FASM 1.73.27 обе ассемблерные программы успешно собраны; полученные для задания 2 результаты совпали с программой на C: `16` для `a=8, b=2, c=4` и `50` для `a=-5, b=3, c=2`. Делитель `c=0` проверен отдельно — обе версии завершаются с кодом ошибки `1`.